- Pyro-SDIS RF-DETR-L resume | Kaggle | 2 GPU T4
  - Mục đích: chạy tiếp 12 epoch còn lại (8/20 → 20/20) từ `checkpoint_7.ckpt` của run GB10 `rfdetr_large_pyro_sdis_gb10`
  - Kaggle Input cần: dataset `pyro-sdis-yolo` (layout `images/{train,val}`, `labels/{train,val}`) + checkpoint resume `.ckpt`; nếu resume qua session mới, đặt `metrics.csv` cạnh checkpoint để giữ history
  - Protocol 13f khóa: 1280 px, 20 epoch, seed 20260707, effective batch 16, augmentation mặc định RF-DETR (không override), không early stop
  - Sai khác bắt buộc so với GB10 (đổi hardware): devices 1→2 (micro batch 4×accum4 → 2/GPU×2GPU×accum4, effective giữ 16); amp `auto`(bf16) → `fp16` (T4 không hỗ trợ bf16); num_workers 4→2
  - Cảnh báo: args nhúng trong checkpoint GB10 cho thấy run gốc chạy `early_stopping=true` (callback `RFDETREarlyStopping` nằm trong ckpt) — notebook này ép `early_stopping=False` theo policy đủ 20 epoch; Lightning sẽ warning bỏ qua state callback đó, chấp nhận được
  - Giữ nguyên GB10: constructor `RFDETRLarge(resolution=1280)` không truyền num_classes (tự suy 1 class từ data.yaml); không override lr_scheduler/warmup (mặc định `step`, lr_drop 100 → LR không drop trong 20 epoch)
  - Shim dataset: notebook tự tạo symlink layout `<split>/<images|labels>` + `data.yaml` (nc:1 smoke) giống `ensure_shim_dataset()` của `train_rfdetr.py` cũ
  - Resume: tự quét working + Kaggle Input, chọn checkpoint full-state epoch cao nhất; đủ 20/20 thì không train lại; nếu OOM → hạ MICRO_BATCH=1, GRAD_ACCUM=8
  - Output: `/kaggle/working/runs/rfdetr_large_pyro_sdis`


In [ ]:
!find /kaggle/input -maxdepth 10 -type d | head -100


In [ ]:
from pathlib import Path
import shutil

SEED = 20260707
EPOCHS = 20
RESOLUTION = 1280
MICRO_BATCH = 2
GRAD_ACCUM = 4
INPUT_BASE = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
RUN_NAME = 'rfdetr_large_pyro_sdis'
RUN_DIR = WORK_ROOT / 'runs' / RUN_NAME
SHIM_DIR = WORK_ROOT / 'pyro-sdis-yolo-rfdetr'

# Xoá thư mục rác do chạy lỗi trước đó
shutil.rmtree(SHIM_DIR, ignore_errors=True)

dataset_candidates = sorted({path.parent.resolve() for path in INPUT_BASE.rglob('images')
                             if (path / 'train').is_dir() and (path / 'val').is_dir()
                             and (path.parent / 'labels' / 'train').is_dir()
                             and (path.parent / 'labels' / 'val').is_dir()})
assert len(dataset_candidates) == 1, f'Cần đúng 1 dataset, thấy: {dataset_candidates}'
DATA_SRC = dataset_candidates[0]

# Ánh xạ đúng tên thư mục và biến src
for src, dst_name in (('train', 'train'), ('val', 'valid')):
    for sub in ('images', 'labels'):
        dst = SHIM_DIR / dst_name / sub
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.is_symlink():
            dst.symlink_to(DATA_SRC / sub / src, target_is_directory=True)

(SHIM_DIR / 'data.yaml').write_text('nc: 1\nnames:\n  0: smoke\n')

train_count = sum(1 for _ in (SHIM_DIR / 'train' / 'images').iterdir())
# Đếm trong thư mục 'valid' thay vì 'val'
val_count = sum(1 for _ in (SHIM_DIR / 'valid' / 'images').iterdir())

assert (train_count, val_count) == (29537, 4099), f'Lệch audit: train={train_count}, val={val_count}'
print({'data_src': str(DATA_SRC), 'shim': str(SHIM_DIR), 'run_dir': str(RUN_DIR), 'train': train_count, 'val': val_count})


In [ ]:
import subprocess
import sys

subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rfdetr[train]==1.8.3'], check=True)

import importlib.metadata
import torch

devices = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'rfdetr': importlib.metadata.version('rfdetr'), 'devices': devices})
assert len(devices) == 2 and all('T4' in name for name in devices), f'Cần Kaggle 2 GPU T4, hiện có {devices}'


In [ ]:
from collections.abc import Mapping
import json
import os
import shutil

PROTOCOL = {'schema': 1, 'framework': 'rfdetr-1.8.3', 'model': 'RFDETRLarge', 'dataset': 'pyro-sdis-yolo', 'resolution': RESOLUTION, 'epochs': EPOCHS, 'seed': SEED, 'micro_batch': MICRO_BATCH, 'grad_accum': GRAD_ACCUM, 'devices': 2, 'strategy': 'ddp_notebook', 'amp_dtype': 'fp16', 'augmentation_policy': 'framework-default-no-user-overrides', 'source_run': 'rfdetr_large_pyro_sdis_gb10-epoch0-7-devices1-batch4-accum4-amp-auto'}
protocol = RUN_DIR / 'resume_protocol.json'
existed = RUN_DIR.exists()

def valid_checkpoint(path, source):
    state = torch.load(path, map_location='cpu', weights_only=False)
    if not isinstance(state, Mapping) or not isinstance(state.get('epoch'), int) or not isinstance(state.get('global_step'), int):
        raise ValueError('thiếu epoch/global_step')
    if not isinstance(state.get('state_dict'), Mapping) or not state['state_dict'] or not state.get('optimizer_states') or not state.get('lr_schedulers') or not isinstance(state.get('loops', {}).get('fit_loop'), Mapping):
        raise ValueError('không full-state')
    if not 7 <= state['epoch'] < EPOCHS:
        raise ValueError(f'epoch ngoài khoảng resume hợp lệ: {state["epoch"]}')
    if source == 'working' and (not protocol.is_file() or json.loads(protocol.read_text(encoding='utf-8')) != PROTOCOL):
        raise ValueError('protocol working không khớp')
    return {'path': path, 'source': source, 'epoch': state['epoch'], 'step': state['global_step']}

def scan_checkpoints(paths, source):
    accepted = []
    for path in paths:
        try:
            accepted.append(valid_checkpoint(path, source))
        except Exception as error:
            print('reject', source, path, error)
    return accepted

if existed and not protocol.is_file():
    raise RuntimeError('working thiếu protocol')

working_checkpoints = scan_checkpoints(RUN_DIR.glob('checkpoint_*.ckpt'), 'working')
input_checkpoints = scan_checkpoints(INPUT_BASE.rglob('checkpoint_*.ckpt'), 'input')

chosen = max(working_checkpoints + input_checkpoints, key=lambda item: (item['epoch'], item['step'], item['source'] == 'working'), default=None)
assert chosen, 'Notebook này chỉ để resume — cần checkpoint_7.ckpt (hoặc mới hơn) trong Kaggle Input hoặc working'

if chosen['source'] == 'input':
    destination = RUN_DIR / 'input_epoch_{:03d}_step_{:09d}.ckpt'.format(chosen['epoch'], chosen['step'])
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix('.tmp')
    shutil.copyfile(chosen['path'], temporary)
    os.replace(temporary, destination)
    resume_path = destination
else:
    resume_path = chosen['path']

history_path = RUN_DIR / f"metrics_through_epoch_{chosen['epoch']:02d}.csv"
metrics_source = RUN_DIR / 'metrics.csv'
if not metrics_source.is_file() and chosen['source'] == 'input':
    metrics_source = chosen['path'].with_name('metrics.csv')
if metrics_source.is_file() and not history_path.is_file():
    shutil.copyfile(metrics_source, history_path)

if not protocol.is_file():
    protocol.write_text(json.dumps(PROTOCOL, indent=2, sort_keys=True), encoding='utf-8')

run_complete = chosen['epoch'] >= EPOCHS - 1
print({'resume_source': chosen['source'], 'resume_path': str(resume_path), 'epoch': chosen['epoch'], 'complete': run_complete, 'metrics_history': str(history_path) if history_path.is_file() else None})


In [ ]:
from rfdetr import RFDETRLarge

if run_complete:
    print(f'Không train lại: checkpoint đã hoàn thành epoch {EPOCHS}')
else:
    model = RFDETRLarge(resolution=RESOLUTION)
    model.train(dataset_dir=str(SHIM_DIR), dataset_file='yolo', output_dir=str(RUN_DIR), epochs=EPOCHS, batch_size=MICRO_BATCH, grad_accum_steps=GRAD_ACCUM, accelerator='gpu', devices=2, strategy='ddp_notebook', amp_dtype='fp16', num_workers=2, checkpoint_interval=1, seed=SEED, early_stopping=False, tensorboard=False, resume=str(resume_path))


In [ ]:
checkpoints = sorted(RUN_DIR.glob('checkpoint_*.ckpt'))
weights = sorted(RUN_DIR.glob('*.pth'))
csvs = sorted(RUN_DIR.glob('metrics*.csv'))
for path in checkpoints + weights + csvs:
    print(path, path.stat().st_size)
assert checkpoints
assert weights
